# ICS 604: APPLIED DATA SCIENCE

## Hypothesis Testing: Common Tests
- ### $\chi^2$ Test
  - #### Goodness-of-Fit Test
  - #### Test of Independence (Test of Association)
- ### $z$-Test
- ### $t$-Test
---

<div style="background-color: #ffffcc; padding: 15px; border-left: 5px solid #ffd700; color: #333;"> 
<strong>Exercise:</strong><br>
    Complete the exercise (<strong>exercise4_chi_squired_tests.ipynb</strong>) and submit the file to <strong>Exercises</strong> under <strong>Assignments</strong> in Lamaku by <strong>Monday, March 30</strong>. This exercise will not be graded but will count toward class participation.
</div>

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats
import matplotlib.pyplot as plt

## The $\chi^2$ Goodness-of-Fit Test

The $\chi^2$ (chi-square) goodness-of-fit test is a statistical hypothesis testing method used to evaluate how well observed data align with what we would expect under a specific theoretical distribution. As one of the oldest and most widely used hypothesis tests, it provides a systematic way to assess whether differences between observed and expected frequencies are due to random chance or indicate a meaningful discrepancy. In practice, the test compares counts across one or more categories to determine if the observed pattern deviates significantly from expectations.

This test is particularly useful when working with categorical data, where observations are grouped into distinct categories rather than measured on a continuous scale. The goodness-of-fit version of the $\chi^2$ test focuses specifically on determining whether the distribution of observed frequencies across these categories matches a predefined set of expected frequencies. If the differences between observed and expected counts are sufficiently large, the test may lead us to reject the null hypothesis, suggesting that the data do not follow the assumed distribution.

Categorical data, also known as nominal data, consist of labels or names used to classify observations into separate groups without any inherent ordering. This type of measurement assigns items to discrete categories, much like an `enum` in computer science, where each value represents a distinct, non-overlapping group. Examples include categories such as colors, types of products, or survey responses. Because these categories are qualitative rather than numerical, statistical methods like the $\chi^2$ goodness-of-fit test are especially well-suited for analyzing them.

### The Card Randomness Test

The card randomness test provides a concrete example of how we can use statistical methods to evaluate whether human choices behave randomly. In this scenario, 200 people are asked to mentally select two cards in sequence from an imaginary deck, and we focus on the suit of the second card. The key question is whether this second choice is truly random and independent of the first, or whether people exhibit hidden biases. The observed counts — 35 clubs, 51 diamonds, 64 hearts, and 50 spades — suggest that some suits may be preferred over others, but we need a formal test to determine whether these differences are statistically meaningful or simply due to chance.

To better understand and evaluate such data, we can think about how to reproduce similar observations through simulation. The observed frequencies can be represented as a vector $O = (O_1, O_2, O_3, O_4) = (35,51,64,50)$, corresponding to the four suit categories. In Python, we might simulate this process by randomly sampling 200 outcomes from four equally likely categories. This setup is a direct generalization of a coin flip: instead of two possible outcomes, we now have four. The appropriate model for this type of experiment is a multinomial distribution, which describes the probabilities of counts across multiple categories when each trial results in exactly one of several possible outcomes.

In [ ]:
size = 200
prob = [1/4, 1/4, 1/4, 1/4]
np.random.multinomial(size, prob)

<br>
The $\chi^2$ goodness-of-fit test provides a formal framework for evaluating whether the observed distribution matches what we would expect under a specific hypothesis. In this case, the null hypothesis ($H_0$) states that all four suits are equally likely, each with probability 0.25, while the alternative hypothesis ($H_A$) claims that at least one suit has a different probability. To test this, we conceptually repeat the process under the assumption that the null hypothesis is true: we simulate many samples of 200 uniformly distributed suit choices, compute a test statistic that measures how far each sample deviates from the expected counts, and then compare our observed statistic to this reference distribution.

This approach mirrors the manual hypothesis testing process: generate data under the null hypothesis, calculate a statistic that captures deviation from expectation, and determine how extreme the observed data are in that context. The $\chi^2$ goodness-of-fit test automates this reasoning by providing a standard test statistic and known reference distribution. If the observed frequencies differ too greatly from the expected uniform distribution $P=(0.25,0.25,0.25,0.25)$, we may reject the null hypothesis in favor of the alternative, concluding that the suit choices are not uniformly random.

### The “Goodness of Fit” Test Statistic

To carry out a goodness-of-fit test, we need a test statistic that quantifies how well our observed data align with what we would expect under the null hypothesis. The goal of this statistic is to measure the “distance” between what we observed and what the null model predicts. If the observed data closely resemble the expected pattern, then the null hypothesis remains plausible. However, if the observed data differ substantially from what we would expect, that provides evidence against the null hypothesis. In the card example, assuming all suits are equally likely, the expected frequencies are obtained by multiplying the total number of observations (200) by the probability of each suit (0.25), yielding equal expected counts for all categories.

In [ ]:
probabilities = pd.Series([0.25, 0.25, 0.25, 0.25], 
                          index=["clubs", "diamonds", "hearts", "spades"])

In [ ]:
expected = 200 * probabilities
expected

#### Compare Expected vs. Observed

The next step is to compare these expected values $E_i$ with the observed values $O_i$ for each category. A natural way to measure the discrepancy is to look at the difference $O_i - E_i$.

<center><img src="https://www.dropbox.com/scl/fi/y73utlbzw5s8i8cikcul0/obs_exp.png?rlkey=lepyq7v3svwypbla7p8ou7w1h&st=rnfi1s2d&dl=1" width="400"></center><br>

However, since positive and negative differences would cancel each other out, we square these differences to ensure all deviations contribute positively. Additionally, not all differences should be treated equally: a difference of 10 is more significant when the expected count is 20 than when it is 200. To account for this, we scale each squared difference by dividing by the expected value $E_i$. This allows us to interpret deviations relative to the size of what we expected, and then we sum these contributions across all categories to obtain a total measure of discrepancy.

This leads to the definition of the $\chi^2$ statistic:

$$
\chi^2 = \sum_{i=1}^{k} \frac{(O_i - E_i)^2}{E_i}
$$

where $k$ is the number of categories. This statistic behaves in an intuitive way: small values indicate that the observed and expected counts are close, suggesting that the null hypothesis is reasonable. Large values indicate substantial deviations, providing evidence that the observed data likely did not come from the hypothesized distribution. In this sense, the $\chi^2$ statistic plays a similar role to distance measures used in simulation-based testing, but it is standardized and widely applicable.

In [ ]:
observed = pd.Series([35, 51, 64, 50], index=["clubs", "diamonds", "hearts", "spades"])
observed - expected

In [ ]:
(observed - expected)**2 / expected

In [ ]:
### Goodness of fit statistic
sum((observed - expected)**2 / expected)

<br>
Finally, interpreting a specific value such as $\chi^2 = 8.44$ requires context. On its own, the number does not tell us much — we need to compare it to the distribution of $\chi^2$ values we would expect under the null hypothesis. Note that the goodness-of-fit test is always one-sided: we are only concerned with whether the statistic is “too large,” since larger values indicate greater disagreement with the null hypothesis.

### The Sampling Distribution of the $\chi^2$ Statistic

To understand how to interpret a computed $\chi^2$ value, we need to know what values of the statistic typically look like when the null hypothesis is true. In other words, we are interested in the *sampling distribution* of the $\chi^2$ statistic under the assumption that the data are generated from the hypothesized distribution (in this case, the uniform distribution over suits). This distribution tells us what kinds of deviations from expectation are normal due to randomness, and which ones are unusually large. Each observed count $O_i$ can be thought of as coming from a binomial distribution with parameters $N$ and $P_i$, since it represents the number of times a particular category occurs in repeated trials.

$$
O_i \sim \text{Binomial}(N, P_i)
$$

When the sample size $N$ is large and the category probabilities $P_I$ are not too extreme, the Central Limit Theorem tells us that these binomially distributed counts are well approximated by a normal (Gaussian) distribution.

In [ ]:
N = 200
p = 0.4

In [ ]:
np.random.binomial(N, p)

In [ ]:
x_axis = np.arange(45, 115)
y_axis = scipy.stats.binom.pmf(x_axis, N, p)

plt.figure(figsize=(8, 4))
plt.plot(x_axis, y_axis);

### From Gaussian to the $\chi^2$

As mentioned above, according to the Central Limit Theorem, the binomially distributed counts are well approximated by a normal (Gaussian) distribution. This is important because it allows us to analyze the behavior of the components of the $\chi^2$ statistic. Specifically, the standardized quantity $(O_i - E_i) / \sqrt{E_i}$ behaves approximately like a standard normal variable, since $E_i = N \times P_i$ is just a constant under the null hypothesis.

The $\chi^2$ statistic is formed by squaring these standardized differences and summing them across all categories:

$$
\chi^2 = \sum_{i=1}^{k} \frac{(O_i - E_i)^2}{E_i}
$$

A key result from probability theory is that the sum of squared independent standard normal variables follows a $\chi^2$ distribution. This explains why the $\chi^2$ statistic has its name and why it follows a known theoretical distribution under the null hypothesis. This distribution appears in many areas of statistics, especially in tests involving variance and categorical data.

An important feature of the $\chi^2$ distribution is its **degrees of freedom**, which determine its shape. In the goodness-of-fit setting with $k$ categories, the degrees of freedom are typically $k - 1$. This reflects the fact that not all category counts are independent: once we know the counts in $k - 1$ categories, the final one is determined because the total number of observations is fixed. Although this explanation captures the intuition, the concept of degrees of freedom can be more subtle in general settings, and in some statistical models it may not even be a whole number.

In [ ]:
scipy.stats.chi2(df=10).rvs()

In [ ]:
scipy.stats.chi2(df=100).rvs(4)

In [ ]:
x_axis = np.arange(0, 40, 0.1)
y_axis = scipy.stats.chi2(df=5).pdf(x_axis)
y2_axis = scipy.stats.chi2(df=10).pdf(x_axis)
y3_axis = scipy.stats.chi2(df=15).pdf(x_axis)

plt.figure(figsize=(4.5, 3))
plt.plot(x_axis, y_axis, label="df = 5")
plt.plot(x_axis, y2_axis, label="df = 10")
plt.plot(x_axis, y3_axis, label="df = 15")
plt.legend();

### Finding the Rejection Region

To decide whether to reject the null hypothesis, we need to determine which values of the $\chi^2$ statistic are considered “too large” to be consistent with the null model. This set of values is called the rejection region. The idea is rooted in the behavior of the sampling distribution: under the null hypothesis, most $\chi^2$ values will fall within a typical range, while unusually large values indicate that the observed data deviate more than we would expect by chance alone.

As discussed earlier, small values of $\chi^2$ mean that the observed counts $O_i$ are very close to the expected counts $E_i$, which supports the null hypothesis. In contrast, large values of $\chi^2$ indicate substantial discrepancies between observed and expected frequencies, suggesting that the data are unlikely to have come from the hypothesized distribution. Because of this, the rejection region is always located in the right tail of the $\chi^2$ distribution.

To formally define this region, we choose a significance level (commonly denoted $\alpha$, such as 0.05). This represents the probability of rejecting the null hypothesis when it is actually true. We then find the critical value from the $\chi^2$ distribution (with the appropriate degrees of freedom) such that only an $\alpha$ proportion of the distribution lies to the right of it. If the computed $\chi^2$ statistic exceeds this critical value, it falls in the rejection region, and we reject the null hypothesis. Otherwise, we do not have sufficient evidence to reject it.

In [ ]:
x_axis = np.arange(0, 20.1, 0.1)
y_axis = scipy.stats.chi2(df=3).pdf(x_axis)

plt.figure(figsize=(4.5, 3))
plt.plot(x_axis, y_axis);

In [ ]:
chi2_rv = scipy.stats.chi2(df=3)

### The cdf refers to the cumulative distribution function.
### The cumulative distribution function (cdf) is the probability 
### that the variable takes a value less than or equal to x. 

chi2_rv.cdf(0)

In [ ]:
chi2_rv.cdf(20)

In [ ]:
print(x_axis[:10])
print(x_axis[-10:])

In [ ]:
probs = chi2_rv.cdf(x_axis)
print(np.round(probs[:10], decimals=4))
print(np.round(probs[-10:], decimals=4))

In [ ]:
np.where(probs > 0.95)

In [ ]:
index = np.where(probs > 0.95)[0][0]
index

In [ ]:
print(x_axis[index])
print(probs[index])

In [ ]:
# ppf (percent point function) is the inverse of the cdf. 
# We can find a critical value using ppf.
# If the test statistic > critical value, reject the null, otherwise retain the null.

chi2_rv.ppf(.95)   # significance level of alpha = 0.05

In [ ]:
# Give me the area under the upper tail instead
1 - chi2_rv.cdf(8.44)

In [ ]:
observed

In [ ]:
# calculate chi-square test
result = scipy.stats.chisquare(observed)
result

In [ ]:
### Specifying different null hypothesis
result = scipy.stats.chisquare(f_obs=observed, f_exp=[40, 50, 60, 50])
result

In [ ]:
print("chi-squared statistic =", result[0])
print("p-value =", round(result[1], 4))

In [ ]:
print("chi-squared statistic =", result.statistic)
print("p-value =", round(result.pvalue, 4))

## The $\chi^2$ Test of Independence (or Association)

The $\chi^2$ test of independence (also called a test of association) extends the goodness-of-fit framework to situations involving two categorical variables. In this example, the goal is to determine whether the site where coral species are introduced (Site A or Site B) is related to their outcomes — whether they live, die, or become unhealthy. After introducing 180 coral samples across the two sites (87 in Site A and 93 in Site B), we summarize the results in a contingency table. The central question is whether the distribution of outcomes is the same across both sites, or whether one site appears more conducive to coral health.

<center><img src="https://www.dropbox.com/scl/fi/a4qeknb8jhj3ggmluwggs/xtab.png?rlkey=7k2w9xxawkhwpvxk02eidvze7&st=5t2ocszx&dl=1" width="500"></center>

### Constructing the Hypothesis Test

To formalize this, we construct hypotheses about the relationship between site and outcome. The null hypothesis ($H_o$) states that the outcome distribution is identical across both sites — in other words, the probability of each outcome (dies, unhealthy, lives) is the same regardless of location. This implies that the two variables (site and outcome) are independent. The alternative hypothesis ($H_A$) is that at least one of these probabilities differs between the two sites, indicating an association between site and coral health outcomes.

We organize the data using a contingency table, where each cell $O_{ij}$ represents the observed count for outcome $i$ at site $j$. The row totals $R_i$ give the total number of observations for each outcome across both sites, while the column totals $C_j$ represent the total number of observations at each site. The grand total is denoted by $N$. Unlike the goodness-of-fit test, where expected probabilities are specified, here we estimate the overall probabilities of each outcome directly from the data. Specifically, the estimated probability of outcome $i$ is given by $\hat{P}_i = R_i/N$, which reflects the proportion of all observations that fall into that category.

<center><img src="https://www.dropbox.com/scl/fi/oxc2iijus0m7z2r03wfad/xtab_generic.png?rlkey=pk5jgfma5ap8gfft33cps18f9&st=bd5fffsl&dl=1" width="400"></center>

Using these estimated probabilities, we compute the expected counts under the assumption that site and outcome are independent. The expected frequency for each cell is given by

$$
\hat{E}_{ij} = \frac{R_i}{N} \times C_j
$$

which can also be interpreted as: (row total × column total) divided by the grand total. These expected counts represent what we would anticipate seeing if there were no association between site and outcome. By comparing the observed counts $O_{ij}$ to these expected counts, we can then compute a $\chi^2$ statistic and assess whether the differences are large enough to reject the null hypothesis of independence.

### Defining the Test Statistic

To test for independence between two categorical variables, we define a $\chi^2$ test statistic in a way that closely mirrors the goodness-of-fit case. The key idea remains the same: we compare observed counts to expected counts under the null hypothesis. However, instead of a single set of categories, we now work with a contingency table that has $r$ rows (e.g., outcomes like lives, dies, unhealthy) and $c$ columns (e.g., Site A and Site B). For each cell in the table, we compute how far the observed count $O_{ij}$ deviates from the expected count $\hat{E}_{ij}$, square that difference, and scale it by the expected value. Summing these contributions across all cells gives the overall test statistic:

$$
\chi^2 = \sum_{i=1}^{r}\sum_{j=1}^{c}\frac{(\hat{E}_{ij}-O_{ij})^2}{\hat{E}_{ij}}
$$

This statistic captures the total discrepancy between what we observe and what we would expect if the two variables were independent.

As in the goodness-of-fit setting, the $\chi^2$ statistic follows a $\chi^2$ distribution under the null hypothesis, provided the sample size is sufficiently large and expected counts are not too small. This allows us to determine whether the observed discrepancies are unusually large by comparing the computed statistic to the appropriate reference distribution. Larger values of $\chi^2$ indicate stronger evidence against the null hypothesis, suggesting that the variables are not independent and may be associated.

An important consideration in this setting is the number of degrees of freedom, which determines the shape of the $\chi^2$ distribution. In a contingency table, not all cell counts are independent because the row and column totals impose constraints. Once we know the counts in most of the cells, the remaining ones are determined automatically. As a result, the degrees of freedom are given by $(r−1)(c−1)$, reflecting the number of independent pieces of information available. This value is used when referencing the $\chi^2$ distribution to compute p-values or critical thresholds for the test.

### Assumptions of the Test

The validity of the $\chi^2$ test of independence relies on several key assumptions. First, the expected frequencies in each cell of the table must be sufficiently large to ensure that the approximation to the $\chi^2$ distribution is reasonable. A common rule of thumb is that **each expected count should be at least 5**. For larger contingency tables, it is often recommended that **at least 80% of the expected counts exceed 5** and that **no expected count is zero**. This requirement is linked to the Central Limit Theorem, which underpins why the squared standardized deviations in the $\chi^2$ statistic follow the $\chi^2$ distribution.

Second, the **observations must be independent**. In other words, the outcome for one observation should not influence the outcome for another. For example, in the coral experiment, the survival or death of one species must not affect the probability that another species dies. Violations of independence can distort the test statistic and lead to misleading results, so careful experimental design or data collection is essential to meet this assumption. When both of these conditions are satisfied, the $\chi^2$ test provides a reliable method for assessing whether categorical variables are independent or whether observed counts deviate significantly from expected counts.

## $z$-Test and $t$-Test

The $z$-test and $t$-test are statistical methods used to evaluate whether observed differences are likely due to random chance or reflect a true underlying effect. In both cases, we compare the observed statistic to what we would expect under the null hypothesis of no difference, mirroring the logic of simulation-based hypothesis testing.

Although they serve similar purposes, the two tests differ in their assumptions and typical use cases. The **$t$-test** is primarily used for **inference about means**, especially when the population standard deviation is unknown (which is most common in practice). It accounts for additional uncertainty by using the $t$-distribution, making it particularly appropriate for smaller sample sizes. In contrast, the **$z$-test** is more general and can be used for **means or proportions**, typically when the sample size is large or when the population standard deviation is known. For example, $z$-tests are commonly used for comparing proportions, such as classification accuracy or conversion rates.

These tests are widely used in practical applications. In medicine, for instance, a $t$-test might be employed to determine whether a new drug produces a statistically significant change in patients’ blood pressure compared to a placebo. In business or data science, a $z$-test might be used to compare proportions (e.g., click-through rates) or, in some cases, average values when large samples justify a normal approximation. By providing a formal framework for quantifying uncertainty, both $z$-tests and $t$-tests enable researchers and analysts to assess whether observed effects are statistically significant.

## One Sample $z$-Test

The one-sample $z$-test is used to determine whether the mean of a sample differs significantly from a known population mean when the population standard deviation is known. In this example, the average expression level of gene $X$ in normal corals is given as $\mu = 67.5$ with a standard deviation $\sigma=9.5$. We then collect a sample of 20 coral survivors from a heat wave and measure their gene expression levels:

<center>50 60 60 64 66 66 67 69 70 74 76 76 77 79 79 79 81 82 82 89 </center>
<br>

The biological question is whether these survivors exhibit a different average expression level, which might help explain their survival under extreme conditions.

In [ ]:
new_data = [50, 60, 60, 64, 66, 66, 67, 69, 70, 74, 76, 76, 77, 79, 79, 79, 81, 82, 82, 89]

np.mean(new_data)

<br>
To evaluate this, we compare the sample mean $\bar{X}$ to the known population mean $\mu$. The basic idea is straightforward: if the difference $\bar{X} - \mu$ is close to zero, the sample is consistent with the null hypothesis. However, if this difference is large (either positive or negative), it suggests that the sample may come from a different distribution. The challenge is determining how large is “large enough” to reject the null hypothesis, which is where the concept of standardization becomes essential.

### $z$-Score

Under the null hypothesis, the sampling distribution of the sample mean follows a normal distribution with mean $\mu$ and standard error $SE(\bar{X}) = \sigma/\sqrt{N}$. By converting the observed sample mean into a standardized $z$-score,

$$
z_{\bar{X}} = \frac{\bar{X} - \mu}{SE(\bar{X})} = \frac{\bar{X} - \mu}{\sigma/\sqrt{N}}
$$

we can express the deviation in terms of standard errors. This $z$-score follows a **standard normal distribution** $\mathcal{N}(\mu, \sigma)$, allowing us to assess how extreme the observed value is relative to what we would expect under the null hypothesis.

In this case, the sample mean of the survivors is 72.3, which is noticeably higher than the population mean of 67.5. When standardized, this yields a $z$-score of 2.26. This means the observed sample mean is 2.26 standard errors above the expected mean under the null hypothesis. Since this value falls in the tail of the standard normal distribution, the corresponding p-value is less than 0.05. Therefore, we reject the null hypothesis and conclude that the average gene expression in the surviving corals is significantly different from that of normal corals, suggesting a possible biological link between gene expression and survival.  

<center><img src="https://www.dropbox.com/scl/fi/nckabz70w02wg2f13dep6/one_two_test.png?rlkey=b1r89n32mvp59bbff8dbwvqso&st=6c51jchy&dl=1" width="700"></center>

In [ ]:
sample_mean = np.mean(new_data)
sample_mean

In [ ]:
mu_null = 67.5
sd_true = 9.5

In [ ]:
N = len(new_data)
sem = sd_true / np.sqrt(N)
z_score = (sample_mean - mu_null) / sem
z_score

In [ ]:
import scipy.stats
rv_norm = scipy.stats.norm(0, 1)
1 - rv_norm.cdf(z_score)

In [ ]:
rv_norm.cdf(-z_score)

In [ ]:
rv_norm.cdf(-z_score) + (1 - rv_norm.cdf(z_score))

In [ ]:
scipy.stats.norm.interval(confidence=0.95, loc=sample_mean, scale=sem)

### Assumptions of the $z$-Test

The one-sample $z$-test relies on several important assumptions to ensure that its results are valid and interpretable. One key assumption is **normality**: the test assumes that the underlying population distribution is normal. This guarantees that the sampling distribution of the mean is also normal. In practice, for large sample sizes, this assumption can be relaxed somewhat due to the Central Limit Theorem, but for smaller samples, normality is especially important.

Another critical assumption is **independence** of observations. Each data point in the sample should be collected independently of the others, meaning that the value of one observation does not influence or provide information about another. Violations of independence — such as repeated measurements on the same subject without proper adjustment — can lead to misleading conclusions because they distort the variability captured by the test.

Finally, the $z$-test assumes that the **population standard deviation is known**. This is a strong assumption and is often unrealistic in real-world scenarios, since we typically only have access to sample data. When the population standard deviation is unknown, which is more common, the $t$-test is generally used instead, as it accounts for the additional uncertainty in estimating variability from the sample.

## One Sample $t$-Test

The one-sample $t$-test is a natural extension of the $z$-test that is used when the population standard deviation is unknown, which is the more common situation in practice. Instead of relying on a known $\sigma$, we estimate the population variability using the sample standard deviation $\hat{\sigma}$. However, this introduces additional uncertainty, since $\hat{\sigma}$ is itself subject to sampling variability. The $t$-test accounts for this extra uncertainty by adjusting both the test statistic and the reference distribution used for inference.

The test statistic is constructed in a similar way to the $z$-score, but replaces the true standard deviation with its sample estimate:

$$
t = \frac{\bar{X} - \mu}{\hat{\sigma} /\sqrt{N}}
$$

This statistic measures how many estimated standard errors the sample mean $\bar{X}$ is away from the hypothesized population mean $\mu$. If the null hypothesis is true, we would expect this value to be close to zero, while large positive or negative values provide evidence against the null hypothesis.

Because we are estimating the standard deviation from the data, the sampling distribution of this statistic is no longer standard normal. Instead, it follows a **$t$-distribution with $N−1$ degrees of freedom**. The $t$-distribution is similar to the normal distribution but has heavier tails, reflecting the additional uncertainty introduced by estimating $\sigma$. As the sample size increases, this uncertainty diminishes, and the $t$-distribution gradually approaches the standard normal distribution.

In [ ]:
res = scipy.stats.ttest_1samp(new_data, popmean=67.5)
print("statistic:", round(res.statistic.item(), 3))
print("pvalue   :", round(res.pvalue.item(), 3))
print("df       :", round(res.df.item(), 1))

In [ ]:
conf_level = 0.95
df = len(new_data) - 1
sample_mean = np.mean(new_data)
sem = scipy.stats.sem(new_data)

conf_interval = scipy.stats.t.interval(conf_level, df, sample_mean, sem)
tuple(round(x.item(), 2) for x in conf_interval)

## The Independent Samples $t$-Test (Student's $t$-Test)

The independent samples $t$-test is used to compare the means of two independent groups of continuous values. It addresses the question: Are the two samples drawn from populations with the same mean, or do they differ? The hypotheses are formulated as:

- Null hypothesis ($H_0$): $\mu_1 = \mu_2$ — the population means are equal
- Alternative hypothesis ($H_A$): $\mu_1 \ne \mu_2$ — the population means differ
               
<center><img src="https://www.dropbox.com/scl/fi/9po93krurxdremkv9jzrl/independent.png?rlkey=dsd720ivttoh9nyspg3v67pg4&st=wsbhj7r5&dl=1" width="500"></center>

The figure above illustrates two independent samples and the potential difference between their means. 

This type of test is commonly used in situations where we want to compare outcomes between two distinct groups, such as treatment vs. control.

### Pooled Estimate of the Standard Deviation

A key assumption of the standard independent $t$-test is that both populations have the **same standard deviation**, i.e., $\sigma_1$ = $\sigma_2 = \sigma$. Under this assumption, we compute a **pooled standard deviation** as a weighted average of the sample variances:

$$
\hat\sigma = \sqrt{\frac{\left(N_1 - 1\right)\hat\sigma_1^2 + \left(N_2 - 1\right)\hat\sigma_2^2}{N_1 + N_2 - 2}}
$$

This pooled estimate provides a single standard deviation to use in the calculation of the test statistic.

### Test Statistic

The independent samples $t$-statistic measures the difference between sample means relative to the variability of the samples:

$$
t = \frac{\bar{X_1} - \bar{X_2}}{SE(\bar{X_1} - \bar{X_2})}
$$

where the standard error of the difference is calculated using the pooled standard deviation:

$$
SE(\bar{X_1} - \bar{X_2})  = \hat{\sigma}\sqrt{\frac{1}{N_1}+ \frac{1}{N_2}}
$$

Under the null hypothesis, this $t$-statistic follows a $t$-distribution with $N_1 + N_2 - 2$ (i.e., $N - 2$) degrees of freedom. By comparing the observed $t$-value to the theoretical $t$-distribution, we can determine whether the difference between sample means is statistically significant.

### Assumptions of the Student's $t$-Test

For the Student’s $t$-test to produce valid results, three key assumptions must be met:

- **Normality:** The values in each group are approximately normally distributed. This is especially important for small sample sizes; for larger samples, the Central Limit Theorem can mitigate minor deviations.
- **Independence:** Observations within each group and between groups are independent. No observation should influence another.
- **Homogeneity of variance (Homoscedasticity):** The population standard deviations in the two groups are assumed to be equal ($\sigma_1 = \sigma_2$). This justifies using the pooled standard deviation in the calculation of the test statistic.

Violations of these assumptions can affect the accuracy of the $t$-test, potentially leading to incorrect conclusions about the equality of means.

In [ ]:
np.random.seed(142)
data_1 = np.random.normal(1, 3, 20)
data_2 = np.random.normal(1, 3, 50)

In [ ]:
# Student's t-test
res = scipy.stats.ttest_ind(data_1, data_2)
print("statistic:", round(res.statistic.item(), 4))
print("pvalue   :", round(res.pvalue.item(), 4))
print("df       :", round(res.df.item(), 1))

## The Independent Samples $t$-Test (Welch's $t$-Test)

A key motivation for using Welch’s $t$-test instead of the classical Student’s $t$-test is that the assumption of equal variances between the two groups is often unrealistic. In real data, different groups frequently exhibit different levels of variability. Welch’s test removes this assumption of homogeneity of variance, making it more flexible and robust. As a result, the primary assumptions that remain are that the data in each group are **approximately normally distributed** and that the observations are **independent**.

The test statistic is constructed by taking the difference between the sample means and scaling it by the standard error of that difference:

$$
t = \frac{\bar{X_1} - \bar{X_2}}{SE(\bar{X_1} - \bar{X_2})}.
$$

Here, the standard error reflects the combined variability of both samples and is calculated as

$$
SE(\bar{X_1} - \bar{X_2}) = \sqrt{\frac{\hat{\sigma}_1^2}{N_1}+ \frac{\hat{\sigma}_2^2}{N_2}}.
$$

This formulation accounts for the fact that each sample contributes its own uncertainty, based on its variance and size.

Once the $t$ statistic is computed, it is compared against a $t$-distribution with adjusted degrees of freedom (specific to Welch’s test). Large positive or negative values of the statistic indicate that the difference in sample means is unlikely to have occurred by chance, providing evidence against the null hypothesis.

In [ ]:
data_1 = np.random.normal(4, 0.8, 20)
data_2 = np.random.normal(1, 1.2, 50)

In [ ]:
# Welch's t-test
res = scipy.stats.ttest_ind(data_1, data_2, equal_var=False)
print("statistic:", round(res.statistic.item(), 4))
print("pvalue   :", round(res.pvalue.item(), 4))
print("df       :", round(res.df.item(), 1))